# FiCO₂ ⇌ PaCO₂ Conversion
### A ventilation-aware converter for CO₂ challenges

*Supplementary tool for the isometabolism review (BOLD-MRI / CMRO₂ during hypercapnia).*

CO₂ challenges are reported in two incompatible currencies. Inhalational and
fixed-inspired-gas protocols specify the **inspired fraction** (FiCO₂, e.g. "5% CO₂");
end-tidal targeting, rebreathing and most imaging cerebrovascular-reactivity work specify a
**partial pressure** (PaCO₂ ≈ PetCO₂, mmHg or kPa). To pool the *magnitude* of a hypercapnic
challenge across studies, the two must be placed on one scale. This notebook does that in
both directions, and — crucially — accounts for the ventilatory feedback that a naïve
conversion ignores. Every constant and equation is referenced; all sources are in the
accompanying Zotero library (**CO2 Ventilation** collection).

## 1. Where the equation comes from

The relationship between ventilation and alveolar gas is one of the oldest quantitative
results in respiratory physiology. Three classical ideas combine into the equation we use.

**(a) Ventilation sets alveolar CO₂.** Haldane & Priestley (1905) showed that alveolar
CO₂ is tightly regulated and that ventilation is driven almost entirely by CO₂. In steady
state, all metabolically produced CO₂ ($\dot V_{CO_2}$) is exhaled in the alveolar gas, so
the alveolar CO₂ *fraction* is simply output over alveolar ventilation:

$$F_ACO_2 = \frac{\dot V_{CO_2}}{\dot V_A} \qquad (1)$$

**(b) The "ideal" alveolar compartment.** Riley & Cournand (1949) formalised a single,
well-mixed *ideal alveolar air* in which, for healthy lungs, arterial and end-tidal values
equal the alveolar one — the assumption $P_aCO_2 \approx P_ACO_2 \approx P_{ET}CO_2$ that also
underlies modern end-tidal targeting (Slessarev et al., 2007).

**(c) Fraction → partial pressure (Dalton).** By Dalton's law of partial pressures
(Dalton, 1802), each gas exerts a pressure equal to its fraction times the total dry-gas
pressure. In humidified alveolar gas at 37 °C the water vapour ($P_{H_2O}=47$ mmHg) is
subtracted first, so $P_ACO_2 = F_ACO_2\,(P_{atm}-P_{H_2O})$. Applying this to (1) gives the
**classical alveolar ventilation equation** (West, 2012; Lumb & Thomas, 2020):

$$P_ACO_2 = \frac{K\,\dot V_{CO_2}}{\dot V_A}, \qquad K = 0.863 \qquad (2)$$

$K=0.863$ is purely a unit constant: $\dot V_{CO_2}$ is conventionally quoted at STPD and
$\dot V_A$ at BTPS, and $K=\tfrac{310}{273}\times\tfrac{760}{1000}$ carries the temperature and
pressure corrections (Cruickshank & Hirschauer, 2004; West, 2012).

**The gap this tool fills.** Equation (2) was built for *CO₂-free* inspired gas — historically
inspired CO₂ was a nuisance term set to zero — and it treats $\dot V_A$ as fixed. Neither holds
for a CO₂ challenge: the inspirate contains CO₂, *and* the induced hypercapnia stimulates
ventilation, which blows off part of the added CO₂. The conversion below restores both terms.

## 2. The conversion, step by step

### Step 1 — Get baseline alveolar ventilation from the baseline PaCO₂

We anchor the model to the subject's normocapnic state. Rearranging the classical equation
(2) at baseline gives the baseline alveolar ventilation directly from the baseline PaCO₂:

$$\boxed{\;\dot V_{A,\text{base}} = \dfrac{K\,\dot V_{CO_2}}{P_aCO_{2,\text{base}}}\;} \qquad (3)$$

With $\dot V_{CO_2}=200$ mL·min⁻¹ and $P_aCO_{2,\text{base}}=40$ mmHg this is ≈ 4.3 L·min⁻¹, the
textbook resting value (West, 2012; Lumb & Thomas, 2020). Anchoring this way makes the whole
mapping *self-consistent*: at FiCO₂ = 0 it returns the baseline exactly.

> **If the baseline is unknown** (only an FiCO₂ was reported), we instead assume
> $P_aCO_{2,\text{base}} = 40$ mmHg and use the literature resting value
> $\dot V_{A,\text{base}} = 4.2$ L·min⁻¹ (West, 2012; Lumb & Thomas, 2020 — *"typical resting
> values might be ~4 L·min⁻¹"*). The steep CO₂ feedback (Step 3) absorbs the tiny mismatch,
> so FiCO₂ = 0 then returns ≈ 40.04 mmHg.

### Step 2 — Add inspired CO₂ to the mass balance

When the inspirate contains CO₂, the alveolar balance gains an inspired term
(Slessarev et al., 2007):

$$F_ACO_2 = F_iCO_2 + \frac{\dot V_{CO_2}}{\dot V_A}. \qquad (4)$$

Multiplying through by $(P_{atm}-P_{H_2O})$ and applying Dalton's law converts every term to a
partial pressure, giving the inspired-CO₂ form of the alveolar equation (Lumb & Thomas, 2020):

$$P_ACO_2 = \underbrace{F_iCO_2\,(P_{atm}-P_{H_2O})}_{P_iCO_2} + \frac{K\,\dot V_{CO_2}}{\dot V_A}. \qquad (5)$$

### Step 3 — Make ventilation respond to CO₂ (the HCVR)

Ventilation is *not* fixed: elevated PaCO₂ drives it up. Over the near-linear working range
(≈ 40–80 mmHg) this **hypercapnic ventilatory response** is well approximated by a straight
line with slope $S$ (Read, 1967; Hirshman et al., 1975; West, 2012):

$$\dot V_A(P_aCO_2) = \dot V_{A,\text{base}} + S\,(P_aCO_2 - P_aCO_{2,\text{base}}). \qquad (6)$$

We adopt the population-mean slope $S = 2.69$ L·min⁻¹·mmHg⁻¹ from Hirshman et al. (1975)
(44 healthy adults; range 1.16–5.95), consistent with rebreathing slopes of 1.96/2.99/4.04
reported by Read (1967) and with textbook "2–3 L·min⁻¹ per mmHg" (West, 2012). Its
variability — the dominant uncertainty — is explored in §6.

### Step 4 — The governing equation and its two directions

Substituting the CO₂-sensitive ventilation (6) into the balance (5), with
$P_aCO_2 \approx P_ACO_2$, gives the single governing equation:

$$P_aCO_2 = F_iCO_2\,(P_{atm}-P_{H_2O}) + \frac{K\,\dot V_{CO_2}}{\dot V_{A,\text{base}} + S\,(P_aCO_2 - P_aCO_{2,\text{base}})}. \qquad (7)$$

**Forward (PaCO₂ → FiCO₂)** is explicit:

$$F_iCO_2 = \frac{P_aCO_2 - K\dot V_{CO_2}/\dot V_A(P_aCO_2)}{P_{atm}-P_{H_2O}}. \qquad (8)$$

**Reverse (FiCO₂ → PaCO₂)** has PaCO₂ on both sides, but because $\dot V_A$ is *linear* in
PaCO₂, equation (7) is a **quadratic** with an exact physical (upper) root — no iteration:

$$P_aCO_2 = \frac{(S P_iCO_2 - D) + \sqrt{(S P_iCO_2 - D)^2 + 4S(P_iCO_2 D + K\dot V_{CO_2})}}{2S},\quad D \equiv \dot V_{A,\text{base}} - S\,P_aCO_{2,\text{base}}. \qquad (9)$$

## 3. Constants and their sources

| Quantity | Value | Source |
|---|---|---|
| $K$ | 0.863 | STPD→BTPS + fraction→pressure unit constant (West, 2012; Cruickshank & Hirschauer, 2004) |
| $P_{atm}$ | 760 mmHg | sea-level barometric pressure |
| $P_{H_2O}$ | 47 mmHg | saturated vapour pressure at 37 °C (Lumb & Thomas, 2020) |
| $P_{atm}-P_{H_2O}$ | 713 mmHg | dry-gas scaling for Dalton's law (Dalton, 1802) |
| $\dot V_{CO_2}$ | 200 mL·min⁻¹ | resting CO₂ output, range 150–200 (West, 2012; Lumb & Thomas, 2020) |
| $P_aCO_{2,\text{base}}$ | 40 mmHg | normocapnic resting baseline |
| $\dot V_{A,\text{base}}$ | $K\dot V_{CO_2}/P_aCO_{2,\text{base}}$ ≈ 4.3 L·min⁻¹ | self-consistent (Eq. 3) |
| $\dot V_{A,\text{rest}}$ | **4.2 L·min⁻¹** | literature resting alveolar ventilation, used when baseline unknown (West, 2012; Lumb & Thomas, 2020) |
| $S$ | 2.69 L·min⁻¹·mmHg⁻¹ | mean HCVR slope, range 1.16–5.95 (Hirshman et al., 1975) |

Load the reference implementation:

In [ ]:
from fico2_paco2_converter import (
    Params, params_from_baseline, params_resting_default,
    paco2_to_fico2, fico2_to_paco2, fico2_to_paco2_numeric,
    mmhg_to_kpa, kpa_to_mmhg, VA_REST, DEFAULT_BASELINE,
)

# Sanity check: a common '5% CO2' inhalation with no known baseline (resting fallback).
p = params_resting_default()
pa = fico2_to_paco2(5.0, p)
print(f'5% CO2  ->  PaCO2 = {pa:.2f} mmHg = {mmhg_to_kpa(pa):.2f} kPa')
print(f'(PaCO2_base={p.PaCO2_base:.0f} mmHg, VA_base={p.VA_base:.1f} L/min, S={p.S})')

## 4. Direction 1 — target PaCO₂ → FiCO₂

Supply the target (hypercapnic) PaCO₂ and the baseline PaCO₂. Step 1 fixes
$\dot V_{A,\text{base}}$ from the baseline; Eq. (8) then returns the required FiCO₂.

In [ ]:
# --- edit these ---
paco2_target = 50.0   # mmHg, the hypercapnic level to reach
paco2_base   = 40.0   # mmHg, the subject's normocapnic baseline
# -------------------

p = params_from_baseline(paco2_base)
fico2 = paco2_to_fico2(paco2_target, p)
print(f'Step 1: VA_base   = {p.VA_base:.3f} L/min   (= K*VCO2/PaCO2_base)')
print(f'        VA(target) = {p.VA(paco2_target):.3f} L/min')
print(f'PaCO2 {paco2_target:.1f} mmHg  ->  FiCO2 = {fico2:.3f} %'      f'   (PICO2 = {fico2/100*p.Pdry:.1f} mmHg)')

## 5. Direction 2 — FiCO₂ → PaCO₂

If the baseline PaCO₂ is known, set it; otherwise leave `paco2_base = None` for the
literature resting fallback ($P_aCO_{2,\text{base}}=40$, $\dot V_{A,\text{base}}=4.2$ L·min⁻¹).
The reverse solve uses the closed-form root, Eq. (9).

In [ ]:
# --- edit these ---
fico2_percent = 5.0    # inspired CO2, %
paco2_base    = None   # mmHg, or None if unknown  ->  resting fallback
# -------------------

if paco2_base is None:
    p = params_resting_default()
    tag = f'resting fallback (base={p.PaCO2_base:.0f}, VA_base={p.VA_base:.1f} L/min)'
else:
    p = params_from_baseline(paco2_base)
    tag = f'baseline known (base={p.PaCO2_base:.0f}, VA_base={p.VA_base:.3f} L/min)'

pa = fico2_to_paco2(fico2_percent, p)
print(f'mode: {tag}')
print(f'FiCO2 {fico2_percent:.2f} %  ->  PaCO2 = {pa:.2f} mmHg = {mmhg_to_kpa(pa):.3f} kPa')
print(f'rise from baseline: {pa - p.PaCO2_base:+.2f} mmHg')

## 6. Reference table & sensitivity to the HCVR slope $S$

Steady-state PaCO₂ for a fixed inspired CO₂ at sea level (baseline 40 mmHg, $S = 2.69$):

In [ ]:
p = params_from_baseline(40.0)
print(f"{'FiCO2 %':>8} {'PICO2 mmHg':>11} {'PaCO2 mmHg':>11} {'PaCO2 kPa':>10}")
for fi in (0, 2, 5, 7, 8):
    pa = fico2_to_paco2(fi, p)
    print(f'{fi:8.0f} {fi/100*p.Pdry:11.1f} {pa:11.2f} {mmhg_to_kpa(pa):10.2f}')

$S$ is the largest source of uncertainty: Hirshman et al. (1975) give a mean of 2.69 but a
wide individual range (1.16–5.95). Mapping the same FiCO₂ across that range shows how much
the estimate depends on the assumed slope (see also Read, 1967; Mohan & Duffin, 1997;
Pedersen et al., 1999 for methods and the two-component central/peripheral response).

In [ ]:
fico2 = 5.0
print(f'FiCO2 = {fico2}%, baseline 40 mmHg\n')
print(f"{'S':>6} {'PaCO2 mmHg':>11}")
for S in (1.16, 1.96, 2.69, 4.04, 5.95):
    p = params_from_baseline(40.0, S=S)
    print(f'{S:6.2f} {fico2_to_paco2(fico2, p):11.2f}')

In [ ]:
# Optional plot (requires matplotlib):  FiCO2 -> PaCO2 across a range of S.
try:
    import numpy as np, matplotlib.pyplot as plt
    grid = np.linspace(0, 8, 81)
    plt.figure(figsize=(6, 4))
    for S in (1.16, 2.69, 5.95):
        p = params_from_baseline(40.0, S=S)
        plt.plot(grid, [fico2_to_paco2(f, p) for f in grid], label=f'S = {S}')
    plt.xlabel('FiCO$_2$ (%)'); plt.ylabel('PaCO$_2$ (mmHg)')
    plt.title('FiCO$_2$ -> PaCO$_2$ vs HCVR slope'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed - skipping plot (pip install matplotlib)')

## 7. Interactive prompt (optional)

Run the next cell to be *prompted* for values, as in the command-line tool
(`python fico2_paco2_converter.py -i`).

In [ ]:
from fico2_paco2_converter import interactive
interactive()

## 8. Assumptions & limitations

- Ideal alveolar–arterial equilibration: $P_aCO_2 \approx P_ACO_2 \approx P_{ET}CO_2$ — healthy
  lungs, no significant V/Q mismatch or shunt (Riley & Cournand, 1949; Slessarev et al., 2007).
- HCVR treated as **linear** over ≈ 40–80 mmHg (Read, 1967; West, 2012).
- A single **population-mean** slope $S = 2.69$; it will not match any individual, and varies
  with age, sex and body size (Patrick & Howard, 1972; Aitken et al., 1986).
- Published $S$ are usually **minute**-ventilation slopes ($\Delta\dot V_E/\Delta P_aCO_2$); used
  here for the **alveolar** slope, valid while dead-space ventilation changes little.

Best used to **harmonise reported CO₂-challenge magnitudes across studies**, not to predict an
individual subject's PaCO₂.

## References
*(all in the Zotero library — CO2 Ventilation collection)*

1. Dalton, J. (1802). *Experimental Essays on the Constitution of Mixed Gases.* Memoirs of the
   Literary and Philosophical Society of Manchester, 5, 535–602.
2. Haldane, J. S., & Priestley, J. G. (1905). The regulation of the lung-ventilation.
   *The Journal of Physiology, 32*(3–4), 225–266. https://doi.org/10.1113/jphysiol.1905.sp001081
3. Riley, R. L., & Cournand, A. (1949). Ideal alveolar air and the analysis of
   ventilation-perfusion relationships in the lungs. *Journal of Applied Physiology, 1*(12),
   825–847. https://doi.org/10.1152/jappl.1949.1.12.825
4. Nielsen, M., & Smith, H. (1952). Studies on the regulation of respiration in acute hypoxia.
   *Acta Physiologica Scandinavica, 24*(4), 293–313.
   https://doi.org/10.1111/j.1748-1716.1952.tb00847.x
5. Read, D. J. C. (1967). A clinical method for assessing the ventilatory response to carbon
   dioxide. *Australasian Annals of Medicine, 16*(1), 20–32.
   https://doi.org/10.1111/imj.1967.16.1.20
6. Patrick, J. M., & Howard, A. (1972). The influence of age, sex, body size and lung size on
   the control and pattern of breathing during CO₂ inhalation in Caucasians.
   *Respiration Physiology, 16*(3), 337–350. https://doi.org/10.1016/0034-5687(72)90063-1
7. Hirshman, C. A., McCullough, R. E., & Weil, J. V. (1975). Normal values for hypoxic and
   hypercapnic ventilatory drives in man. *Journal of Applied Physiology, 38*(6), 1095–1098.
   https://doi.org/10.1152/jappl.1975.38.6.1095
8. Aitken, M. L., Franklin, J. L., Pierson, D. J., & Schoene, R. B. (1986). Influence of body
   size and gender on control of ventilation. *Journal of Applied Physiology, 60*(6),
   1894–1899. https://doi.org/10.1152/jappl.1986.60.6.1894
9. Mohan, R., & Duffin, J. (1997). The effect of hypoxia on the ventilatory response to carbon
   dioxide in man. *Respiration Physiology, 108*(2), 101–115.
   https://doi.org/10.1016/S0034-5687(97)00024-8
10. Pedersen, M. E. F., Fatemian, M., & Robbins, P. A. (1999). Identification of fast and slow
    ventilatory responses to carbon dioxide under hypoxic and hyperoxic conditions in humans.
    *The Journal of Physiology, 521*(1), 273–287.
    https://doi.org/10.1111/j.1469-7793.1999.00273.x
11. Cruickshank, S., & Hirschauer, N. (2004). The alveolar gas equation. *Continuing Education
    in Anaesthesia Critical Care & Pain, 4*(1), 24–27. https://doi.org/10.1093/bjaceaccp/mkh008
12. Slessarev, M., Han, J., Mardimae, A., Prisman, E., Preiss, D., Volgyesi, G., Ansel, C.,
    Duffin, J., & Fisher, J. A. (2007). Prospective targeting and control of end-tidal CO₂ and
    O₂ concentrations. *The Journal of Physiology, 581*(3), 1207–1219.
    https://doi.org/10.1113/jphysiol.2007.129395
13. West, J. B. (2012). *Respiratory Physiology: The Essentials* (9th ed.). Lippincott Williams
    & Wilkins.
14. Lumb, A. B., & Thomas, C. R. (2020). *Nunn's Applied Respiratory Physiology* (9th ed.).
    Elsevier.